In [1]:
!pip install -q --upgrade git+https://github.com/huggingface/transformers.git accelerate torch safetensors tabulate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip uninstall -y torchvision torchaudio

In [3]:
import gc
import sys
import time
from unittest.mock import MagicMock
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Prevent torchvision import crash in Kaggle environments
for mod in [
    'torchvision',
    'torchvision.io',
    'torchvision.transforms',
    'torchvision.transforms.functional',
    'torchvision.ops',
]:
    sys.modules[mod] = MagicMock()

try:
    import transformers.utils.import_utils as _iu
    _iu._torchvision_available = False
    _iu.is_torchvision_available = lambda *a, **kw: False
except Exception:
    pass

# 2. Model Identifiers & Hardware Setup
BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

print(f"✅ Hardware: {DEVICE} | Precision: {DTYPE}")

# 3. Benchmark Prompts
TEST_PROMPTS = [
    {
        "category": "Python Code Completion",
        "title": "Two Sum with Indices",
        "prompt": 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n',
    },
    {
        "category": "Algorithmic Logic",
        "title": "Reverse Singly Linked List",
        "prompt": 'class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\ndef reverse_list(head: ListNode) -> ListNode:\n    """Reverses a singly linked list in-place and returns the new head."""\n',
    },
    {
        "category": "Fast Computation (Numpy/Vectorized)",
        "title": "Cosine Similarity Matrix",
        "prompt": 'import numpy as np\n\ndef batch_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    """Compute pair-wise cosine similarity between two 2D arrays a (N, D) and b (M, D)."""\n',
    },
    {
        "category": "Math Reasoning (Chain-of-Thought)",
        "title": "Word Problem",
        "prompt": "Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer: Let's think step by step.\n",
    },
    {
        "category": "Docstring to Implementation",
        "title": "Binary Search",
        "prompt": 'def binary_search(arr: list[int], target: int) -> int:\n    """Return index of target in sorted arr, or -1 if not found."""\n',
    }
]

# 4. Evaluation Engine Function
def evaluate_model(
    model_id: str,
    prompts: list,
    max_new_tokens: int = 140,
    temperature: float = 0.6,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
):
    print(f"\n==================================================")
    print(f"Loading Model: {model_id}")
    print(f"==================================================")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    
    results = []
    
    for idx, item in enumerate(prompts):
        prompt_text = item["prompt"]
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        start_t = time.perf_counter()
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True if temperature > 0.0 else False,
                temperature=temperature if temperature > 0.0 else 1.0,
                top_p=top_p if temperature > 0.0 else 1.0,
                repetition_penalty=repetition_penalty,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )
        elapsed = time.perf_counter() - start_t
        
        generated_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
        tok_count = len(generated_tokens)
        speed = tok_count / elapsed if elapsed > 0 else 0
        completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        results.append({
            "category": item["category"],
            "title": item["title"],
            "prompt": prompt_text,
            "completion": completion.strip(),
            "tokens_gen": tok_count,
            "speed": f"{speed:.1f} tok/s",
            "time_sec": round(elapsed, 2)
        })
        print(f"  [{idx+1}/{len(prompts)}] {item['title']}: {tok_count} tokens in {elapsed:.2f}s ({speed:.1f} tok/s)")
        
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return results

# 5. Run Comparisons
print("Starting Benchmark...")
base_results = evaluate_model(BASE_MODEL_ID, TEST_PROMPTS)
cpt_results = evaluate_model(CPT_MODEL_ID, TEST_PROMPTS)
print("\n✅ Benchmark Complete!")

✅ Hardware: cuda | Precision: torch.bfloat16
Starting Benchmark...

Loading Model: Qwen/Qwen3.5-0.8B-Base


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

[transformers] A new version of the following files was downloaded from https://huggingface.co/kaptaan45/QaptaanLM-0.75B:
- modeling_qaptaan.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/250 [00:00<?, ?B/s]

  [1/5] Two Sum with Indices: 140 tokens in 8.58s (16.3 tok/s)
  [2/5] Reverse Singly Linked List: 140 tokens in 8.55s (16.4 tok/s)
  [3/5] Cosine Similarity Matrix: 140 tokens in 8.24s (17.0 tok/s)
  [4/5] Word Problem: 52 tokens in 3.40s (15.3 tok/s)
  [5/5] Binary Search: 140 tokens in 8.13s (17.2 tok/s)

✅ Benchmark Complete!


In [6]:
from IPython.display import HTML, display

html_output = """
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto;">
    <div style="background: linear-gradient(135deg, #1e293b, #0f172a); padding: 20px; border-radius: 12px; border: 1px solid #334155; margin-bottom: 24px; text-align: center;">
        <h2 style="color: #f8fafc; margin: 0 0 8px 0; font-size: 24px;">🚀 Head-to-Head Model Comparison</h2>
        <p style="color: #94a3b8; margin: 0; font-size: 14px;">
            Base Model (<strong>Qwen/Qwen3.5-0.8B-Base</strong>) vs CPT Model (<strong>kaptaan45/QaptaanLM-0.75B</strong>)
        </p>
    </div>
"""

for idx, (b, c) in enumerate(zip(base_results, cpt_results)):
    html_output += f"""
    <div style="border: 1px solid #334155; border-radius: 10px; margin-bottom: 24px; background-color: #1e293b; color: #f8fafc; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.3);">
        <div style="background: #0f172a; padding: 12px 16px; border-bottom: 1px solid #334155; display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #38bdf8; font-weight: bold; font-size: 15px;">#{idx+1}. [{b['category']}] {b['title']}</span>
        </div>
        
        <div style="padding: 14px 16px; background: #182234; border-bottom: 1px solid #334155;">
            <div style="color: #94a3b8; font-size: 12px; font-weight: 600; text-transform: uppercase; margin-bottom: 6px;">Prompt</div>
            <pre style="margin: 0; background: #0b1120; padding: 10px 12px; border-radius: 6px; color: #e2e8f0; font-family: 'Fira Code', monospace; font-size: 12px; white-space: pre-wrap;">{b['prompt']}</pre>
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px; padding: 16px; background: #1e293b;">
            <!-- Base Model Column -->
            <div style="background: #0f172a; padding: 14px; border-radius: 8px; border: 1px solid #1e3a8a; display: flex; flex-direction: column;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                    <span style="color: #60a5fa; font-weight: bold; font-size: 14px;">Qwen3.5-0.8B (Base)</span>
                    <span style="background: #1e3a8a; color: #bfdbfe; font-size: 11px; padding: 2px 8px; border-radius: 12px;">{b['speed']}</span>
                </div>
                <div style="color: #64748b; font-size: 11px; margin-bottom: 8px;">{b['tokens_gen']} tokens in {b['time_sec']}s</div>
                <pre style="margin: 0; flex-grow: 1; background: #020617; padding: 12px; border-radius: 6px; color: #cbd5e1; font-family: 'Fira Code', monospace; font-size: 12px; line-height: 1.5; white-space: pre-wrap; overflow-x: auto;">{b['completion']}</pre>
            </div>
            
            <!-- CPT Model Column -->
            <div style="background: #0f172a; padding: 14px; border-radius: 8px; border: 1px solid #14532d; display: flex; flex-direction: column;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                    <span style="color: #4ade80; font-weight: bold; font-size: 14px;">QaptaanLM-0.75B (CPT)</span>
                    <span style="background: #14532d; color: #bbf7d0; font-size: 11px; padding: 2px 8px; border-radius: 12px;">{c['speed']}</span>
                </div>
                <div style="color: #64748b; font-size: 11px; margin-bottom: 8px;">{c['tokens_gen']} tokens in {c['time_sec']}s</div>
                <pre style="margin: 0; flex-grow: 1; background: #020617; padding: 12px; border-radius: 6px; color: #4ade80; font-family: 'Fira Code', monospace; font-size: 12px; line-height: 1.5; white-space: pre-wrap; overflow-x: auto;">{c['completion']}</pre>
            </div>
        </div>
    </div>
    """

html_output += "</div>"
display(HTML(html_output))

In [5]:
import gc
import sys
import time
from unittest.mock import MagicMock
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import HTML, display

# 1. Hardware setup & data types
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

TEST_PROMPTS = [
    {
        "category": "Python Code Completion",
        "title": "Two Sum with Indices",
        "prompt": 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n',
    },
    {
        "category": "Algorithmic Logic",
        "title": "Reverse Singly Linked List",
        "prompt": 'class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\ndef reverse_list(head: ListNode) -> ListNode:\n    """Reverses a singly linked list in-place and returns the new head."""\n',
    },
    {
        "category": "Fast Computation (Numpy)",
        "title": "Cosine Similarity Matrix",
        "prompt": 'import numpy as np\n\ndef batch_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    """Compute pair-wise cosine similarity between two 2D arrays a (N, D) and b (M, D)."""\n',
    },
    {
        "category": "Math Reasoning (Chain-of-Thought)",
        "title": "Word Problem",
        "prompt": "Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer: Let's think step by step.\n",
    },
    {
        "category": "Docstring to Implementation",
        "title": "Binary Search",
        "prompt": 'def binary_search(arr: list[int], target: int) -> int:\n    """Return index of target in sorted arr, or -1 if not found."""\n',
    }
]

def run_benchmark(model_id: str, prompts: list, max_new_tokens: int = 140):
    print(f"\n========================================================")
    print(f"Loading & Evaluating: {model_id}")
    print(f"========================================================")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    results = []
    for idx, item in enumerate(prompts):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to(DEVICE)
        start_t = time.perf_counter()
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                repetition_penalty=1.15,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - start_t
        gen_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
        tok_count = len(gen_tokens)
        speed = tok_count / elapsed if elapsed > 0 else 0
        completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        results.append({
            "category": item["category"],
            "title": item["title"],
            "prompt": item["prompt"],
            "completion": completion.strip(),
            "tokens_gen": tok_count,
            "speed": f"{speed:.1f} tok/s",
            "time_sec": round(elapsed, 2)
        })
        print(f"  [{idx+1}/{len(prompts)}] {item['title']}: {tok_count} tokens in {elapsed:.2f}s ({speed:.1f} tok/s)")

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

# Run Base & CPT Models
base_res = run_benchmark(BASE_MODEL_ID, TEST_PROMPTS)
cpt_res = run_benchmark(CPT_MODEL_ID, TEST_PROMPTS)

# Render Head-to-Head Comparison Table
html_output = "<h2>🚀 Head-to-Head Comparison (Fast Inference Enabled)</h2>"
for idx, (b, c) in enumerate(zip(base_res, cpt_res)):
    html_output += f"""
    <div style="border: 1px solid #334155; border-radius: 8px; margin-bottom: 20px; padding: 16px; background-color: #1e293b; color: #f8fafc;">
        <h3 style="color: #38bdf8; margin: 0 0 8px 0;">#{idx+1}. [{b['category']}] {b['title']}</h3>
        <pre style="background: #0f172a; padding: 8px; border-radius: 4px; color: #cbd5e1; font-size: 12px;">{b['prompt']}</pre>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 12px;">
            <div style="background: #0f172a; padding: 12px; border-radius: 6px; border-left: 3px solid #60a5fa;">
                <h4 style="color: #60a5fa; margin: 0 0 4px 0;">Base Model (Qwen3.5-0.8B)</h4>
                <small style="color: #94a3b8;">{b['tokens_gen']} tokens | {b['speed']} | {b['time_sec']}s</small>
                <pre style="background: #020617; padding: 8px; border-radius: 4px; color: #cbd5e1; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{b['completion']}</pre>
            </div>
            <div style="background: #0f172a; padding: 12px; border-radius: 6px; border-left: 3px solid #4ade80;">
                <h4 style="color: #4ade80; margin: 0 0 4px 0;">CPT Model (QaptaanLM-0.75B)</h4>
                <small style="color: #94a3b8;">{c['tokens_gen']} tokens | {c['speed']} | {c['time_sec']}s</small>
                <pre style="background: #020617; padding: 8px; border-radius: 4px; color: #4ade80; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{c['completion']}</pre>
            </div>
        </div>
    </div>
    """
display(HTML(html_output))



Loading & Evaluating: Qwen/Qwen3.5-0.8B-Base


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

  [1/5] Two Sum with Indices: 119 tokens in 7.67s (15.5 tok/s)


KeyboardInterrupt: 